# CruzTenant: Automated Eviction Protection & Lease Analyzer with Gemma 4
### Build with Gemma: Cruz Into The Gemmaverse! Hackathon — Track 1 (Autonomous Agent Track)

**CruzTenant** is an autonomous AI agent leveraging **Gemma 4 native function calling** to address Santa Cruz's housing crisis. It evaluates lease agreements, rent increase notices, and eviction threats against Santa Cruz Municipal Code (Chapters 21.03 & 21.04) and California state statutes (AB 1482 & AB 12).

In [ ]:
# 1. Setup & Environment Imports
import json
import re
import datetime
from typing import Dict, Any, List, Optional

print('initializing CruzTenant Gemma 4 agent core...')

In [ ]:
# 2. Santa Cruz Legal Knowledge Engine & CPI Constants
CURRENT_SANTA_CRUZ_CPI = 3.8
AB1482_BASE_CAP = 5.0
MAX_ALLOWED_RENT_INCREASE_PERCENT = AB1482_BASE_CAP + CURRENT_SANTA_CRUZ_CPI

SANTA_CRUZ_MUNICIPAL_CODES = {
    "21.03.010": {
        "title": "Santa Cruz just cause eviction protections",
        "description": "landlords must provide a valid at-fault or no-fault just cause reason to terminate tenancies of 12+ months.",
        "jurisdiction": "City of Santa Cruz"
    },
    "21.03.050": {
        "title": "mandatory relocation assistance for no-fault eviction",
        "description": "landlords terminating tenancies under no-fault cause must pay relocation assistance equal to 2 months rent or $3,000.",
        "min_relocation_usd": 3000
    },
    "21.04.020": {
        "title": "rent stabilization and excessive rent increases",
        "description": "rent increases exceeding 5% + CPI (8.8%) trigger mandatory mediation rights and potential invalidation under City Ordinance.",
        "max_increase_pct": MAX_ALLOWED_RENT_INCREASE_PERCENT
    },
    "CA_AB_12": {
        "title": "California AB 12 security deposit limit",
        "description": "security deposits in CA residential leases cannot exceed one month rent starting July 1, 2024."
    }
}

def calculate_rent_cap(current_rent: float, proposed_rent: float, zip_code: str = '95060') -> Dict[str, Any]:
    dollar_increase = proposed_rent - current_rent
    percent_increase = (dollar_increase / current_rent) * 100.0 if current_rent > 0 else 0.0
    max_allowed_percent = MAX_ALLOWED_RENT_INCREASE_PERCENT
    max_legal_rent = current_rent * (1.0 + (max_allowed_percent / 100.0))
    is_excessive = percent_increase > max_allowed_percent
    excess_monthly = max(0.0, proposed_rent - max_legal_rent)
    
    return {
        "current_rent": current_rent,
        "proposed_rent": proposed_rent,
        "percent_increase": round(percent_increase, 2),
        "max_allowed_percent": round(max_allowed_percent, 2),
        "max_legal_rent": round(max_legal_rent, 2),
        "is_excessive": is_excessive,
        "excess_monthly": round(excess_monthly, 2),
        "statute": "Santa Cruz Municipal Code 21.04.020 & California AB 1482"
    }

def verify_eviction_notice(notice_type: str, lease_duration_months: int, stated_reason: str, relocation_offered: float = 0.0, current_rent: float = 2500.0) -> Dict[str, Any]:
    has_just_cause_protection = lease_duration_months >= 12
    reason_lower = stated_reason.lower()
    is_no_fault = 'renovat' in reason_lower or 'remodel' in reason_lower or 'owner move' in reason_lower or 'ellis' in reason_lower
    required_reloc = max(current_rent * 2.0, 3000.0) if is_no_fault else 0.0
    
    violations = []
    if has_just_cause_protection and not is_no_fault and not any(r in reason_lower for r in ['nonpayment', 'breach', 'nuisance']):
        violations.append('notice fails to state a valid just cause reason under Santa Cruz Municipal Code 21.03.010.')
    if is_no_fault and relocation_offered < required_reloc:
        violations.append(f'offered ${relocation_offered} relocation, but Santa Cruz Municipal Code 21.03.050 requires at least ${required_reloc}.')
        
    return {
        "notice_type": notice_type,
        "is_unlawful": len(violations) > 0,
        "violations": violations,
        "required_relocation": required_reloc
    }

In [ ]:
# 3. Gemma 4 Native Tool Definitions
GEMMA_TOOL_SCHEMAS = [
    {
        "name": "calculate_max_allowed_rent_increase",
        "description": "calculates whether rent hike exceeds Santa Cruz 8.8% cap (5% base + 3.8% CPI).",
        "parameters": {"type": "OBJECT", "properties": {"current_rent": {"type": "NUMBER"}, "proposed_rent": {"type": "NUMBER"}}}
    },
    {
        "name": "verify_just_cause_eviction_notice",
        "description": "validates eviction notice against Santa Cruz Municipal Code 21.03 Just Cause & Relocation rules.",
        "parameters": {"type": "OBJECT", "properties": {"notice_type": {"type": "STRING"}, "lease_duration_months": {"type": "NUMBER"}, "stated_reason": {"type": "STRING"}}}
    }
]

print('Gemma 4 function schemas registered successfully.')

In [ ]:
# 4. Demonstration Run: Analyzing Downtown Santa Cruz Excessive Rent Increase
sample_input = "my current rent on Pacific Ave in Downtown Santa Cruz is $2,800/mo. my landlord served an 18% rent hike notice to $3,304/mo. is this legal?"

rent_check = calculate_rent_cap(current_rent=2800.0, proposed_rent=3304.0)
print('=== GEMMA 4 TOOL EXECUTION RESULT ===')
print(json.dumps(rent_check, indent=2))

print('\n=== CRUZTENANT DIAGNOSIS ===')
print(f"unlawful rent increase detected!")
print(f"- proposed increase: {rent_check['percent_increase']}%")
print(f"- Santa Cruz legal cap: {rent_check['max_allowed_percent']}%")
print(f"- maximum allowable legal rent: ${rent_check['max_legal_rent']:.2f}")
print(f"- monthly overcharge: ${rent_check['excess_monthly']:.2f} / month")